# 무노균병 개선 실험: 256px / 384px 미세조정

동일한 기존 최고 모델에서 시작해 입력 해상도를 비교합니다. 두 실행 모두 **5에폭**, 학습률 **1e-5**, batch **16**, seed **42**, ResNet50입니다. 학습 크롭·반전·회전·색상 증강, 클래스 균형 샘플링과 손실함수는 기존 설정을 유지합니다. 전체 32,513장 학습 / 4,070장 검증을 사용합니다.

- 공통 시작 모델: `cropy_runs/full512_resnet50_v1/best_model.pth` (기존 3에폭 최고 모델).
- 원본 모델의 가중치만 불러오고 옵티마이저·스케줄러는 새로 시작합니다.
- 비교: 256px 대조군과 384px 실험군. 둘 다 AdamW / cosine 5에폭 / weight decay 1e-4 / label smoothing 0.1 / FP32.
- 학습 전(epoch 0)과 5에폭까지의 검증을 저장합니다. 최고 Macro F1은 epoch 0도 포함하며, 별도로 고정된 마지막 5에폭 결과도 비교합니다.
- 결과는 `cropy_runs/resolution_finetune_20261005` 아래에 저장됩니다. 기존 모델과 데이터는 보존합니다.

GPU(T4)에서 **모두 실행**합니다. 중단되면 같은 설정으로 다시 모두 실행하여 마지막 완료 에폭부터 재개합니다. 384px 메모리 점검이 실패하면 두 실행을 시작하지 않습니다. 검증셋을 이미 분석했으므로 독립 테스트 성능으로 해석하지 않습니다.

In [ ]:
from pathlib import Path
import base64, csv, hashlib, io, json, shutil, subprocess, sys, zipfile
from collections import Counter
import torch, torchvision
from google.colab import drive

assert torch.cuda.is_available(), 'GPU(T4) 런타임을 선택하세요.'
drive.mount('/content/drive')
UPLOAD = Path('/content/drive/MyDrive/cropy_full_512')
WORK = Path('/content/cropy_resolution_finetune_20261005')
REFERENCE = Path('/content/drive/MyDrive/cropy_runs/full512_resnet50_v1')
SOURCE = REFERENCE / 'best_model.pth'
RUNS = Path('/content/drive/MyDrive/cropy_runs/resolution_finetune_20261005')
EPOCHS, BATCH_SIZE, LR = 5, 16, 1e-5
EXPECTED_SOURCE_SHA256 = '4ef9988d85259b776eaf8fa5824edd2c28b08371e0c19296221e107b670d0060'
EXPECTED_DATA_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_MANIFEST_SHA256 = '1d8924f17fa8cc02f69607a14a60e879301e24e2fd74a8a01792b491d8701b34'

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

assert SOURCE.is_file() and digest(SOURCE) == EXPECTED_SOURCE_SHA256, '시작 모델이 기존 최고 모델과 다릅니다.'
reference_config = json.loads((REFERENCE / 'config.json').read_text(encoding='utf-8'))
reference_env = json.loads((REFERENCE / 'environment.json').read_text(encoding='utf-8'))
assert str(torch.__version__) == reference_env['torch'], '기존 모델과 PyTorch 버전이 다릅니다.'
assert torch.cuda.get_device_name(0) == reference_env['gpu'], '같은 T4 GPU 환경에서 실행하세요.'
assert reference_config['arch'] == 'resnet50' and reference_config['img_size'] == 256
assert reference_config.get('train_geometry', 'random_crop') == 'random_crop'
assert reference_config['manifest_sha256'] == EXPECTED_MANIFEST_SHA256
assert RUNS.resolve() != REFERENCE.resolve() and not RUNS.resolve().is_relative_to(REFERENCE.resolve())
WORK.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('공통 시작 모델 SHA256:', EXPECTED_SOURCE_SHA256)
print('결과 폴더:', RUNS)


## 데이터 및 코드 검증
기존 512px 이미지 패키지를 재사용하며 36,583장의 이미지와 라벨 해시를 확인합니다.

In [ ]:
EXPECTED_CODE_SHA256 = '012fd7554f2fcee03428ac86688f79c084fa258970ddd64a68c421392fc7ee73'
SOURCE_HASHES = {'train.py': '0e72b4fad3e98f02df46652fd9094f48bc826582221236f779f890979c16f6ff', 'train_full_colab.py': '95214007912dccb3a49791dbe4bdccc9f01c7d4fd2451d27b87e32cafd0de9ae', 'train_group_baseline.py': 'd10243e4a3ebe256b885bafca95f3cabac24692270d54f4ea0959b39340837fc', 'compare_crop_ablation.py': 'db1136e5eeec0816939fe2a0193743237b323d35d8cd4ccd29cb477df9e4303f', 'compare_resolution_finetune.py': 'a68ed1596d7aae611f0501949ea1ab7c7d488b9516aabc898ab9bdf40c89eae3'}
CODE_B64 = '...'

def extract_safely(z, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for member in z.infolist():
        assert (destination / member.filename).resolve().is_relative_to(destination.resolve()), '잘못된 ZIP 경로'
    z.extractall(destination)

source_zip = UPLOAD / 'cropy_full_512.zip'
local_zip = WORK / source_zip.name
assert source_zip.is_file(), '기존 데이터 ZIP이 없습니다.'
if not local_zip.exists() or digest(local_zip) != EXPECTED_DATA_SHA256:
    print('기존 데이터 ZIP을 런타임으로 복사 중...', flush=True)
    shutil.copyfile(source_zip, local_zip)
assert digest(local_zip) == EXPECTED_DATA_SHA256
DATA = WORK / 'dataset'
if not (WORK / 'data_verified.json').exists():
    with zipfile.ZipFile(local_zip) as z:
        assert shutil.disk_usage(WORK).free > sum(i.file_size for i in z.infolist()) + 3_000_000_000
        extract_safely(z, WORK)
blob = base64.b64decode(CODE_B64)
assert hashlib.sha256(blob).hexdigest() == EXPECTED_CODE_SHA256
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    extract_safely(z, WORK / 'code')
for name, expected_hash in SOURCE_HASHES.items():
    assert digest(WORK / 'code' / name) == expected_hash
assert digest(DATA / 'manifest.csv') == EXPECTED_MANIFEST_SHA256
with (DATA / 'manifest.csv').open(encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
assert Counter(r['split'] for r in rows) == {'1.Training': 32513, '2.Validation': 4070}
assert len({r['sha256'] for r in rows}) == len(rows) == 36583
assert len({r['packaged_sha256'] for r in rows}) == len(rows)
for i, row in enumerate(rows, 1):
    assert digest(DATA / row['image_path']) == row['packaged_sha256'], row['image_path']
    assert digest(DATA / row['label_path']) == row['label_sha256'], row['label_path']
    if i % 5000 == 0: print('검증:', i, '/', len(rows), flush=True)
train_ds = torchvision.datasets.ImageFolder(DATA / 'images/train')
val_ds = torchvision.datasets.ImageFolder(DATA / 'images/val')
assert (len(train_ds), len(val_ds)) == (32513, 4070)
assert train_ds.classes == val_ds.classes == reference_config['classes']
(WORK / 'data_verified.json').write_text(json.dumps({'images':len(rows),'manifest_sha256':EXPECTED_MANIFEST_SHA256}))
print('데이터와 코드 해시 검증 통과', flush=True)


## 384px GPU 메모리 사전 점검
임시 모델로 batch 16의 순전파·역전파·옵티마이저 갱신을 점검한 뒤 프로세스를 종료합니다. 실제 시작 체크포인트에는 영향을 주지 않습니다.

In [ ]:
preflight = r"""
import json, torch
from torchvision import models
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
model = models.resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 9)
model.cuda().train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
for _ in range(2):
    optimizer.zero_grad()
    images = torch.randn(16, 3, 384, 384, device='cuda')
    labels = torch.arange(16, device='cuda') % 9
    loss = criterion(model(images), labels)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
print(json.dumps({'preflight':'passed','img_size':384,'batch_size':16,'precision':'float32',
 'max_memory_allocated_bytes':torch.cuda.max_memory_allocated(),
 'max_memory_reserved_bytes':torch.cuda.max_memory_reserved(),
 'gpu':torch.cuda.get_device_name(0)}))
"""
check = subprocess.run([sys.executable, '-u', '-c', preflight], text=True, capture_output=True)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise RuntimeError('384px 메모리 사전 점검 실패. 두 실행을 시작하지 않았습니다.')
memory_check = json.loads(check.stdout.strip().splitlines()[-1])
(RUNS / 'memory_preflight.json').write_text(json.dumps(memory_check, indent=2))
assert digest(SOURCE) == EXPECTED_SOURCE_SHA256


## 대조군과 실험군 순차 실행
256px를 먼저 완료한 뒤 384px를 실행합니다. 각 실행은 같은 시작 모델을 새로 불러옵니다. 각 에폭 끝에 Drive로 체크포인트를 저장합니다. 재실행 시 완료된 에폭은 반복하지 않습니다. 두 실행의 최고 모델과 고정된 5에폭 결과를 모두 비교합니다.

In [ ]:
planned_commands = {}
planned_setups = {}
for size in (256, 384):
    run = RUNS / f'finetune_{size}'
    assert run.resolve() != REFERENCE.resolve()
    assert digest(SOURCE) == EXPECTED_SOURCE_SHA256
    command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
        '--data-dir', str(DATA / 'images'), '--output-dir', str(run),
        '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--num-workers', '2',
        '--img-size', str(size), '--arch', 'resnet50', '--train-geometry', 'random_crop',
        '--init-checkpoint', str(SOURCE), '--lr', str(LR)]
    if (run / 'last_checkpoint.pth').exists():
        command.append('--resume')
    setup = dict(command=command, source_sha256=EXPECTED_SOURCE_SHA256,
        code_sha256=EXPECTED_CODE_SHA256, source_hashes=SOURCE_HASHES,
        data_sha256=EXPECTED_DATA_SHA256, torch=str(torch.__version__),
        torchvision=str(torchvision.__version__), gpu=torch.cuda.get_device_name(0))
    setup_path = RUNS / f'setup_{size}.json'
    if setup_path.exists():
        previous = json.loads(setup_path.read_text(encoding='utf-8'))
        for key in setup.keys() - {'command'}:
            assert previous.get(key) == setup[key], f'{size}px 재개 출처 불일치: {key}'
        assert [v for v in previous['command'] if v != '--resume'] == [v for v in command if v != '--resume'], '재개 학습 명령이 다릅니다.'
    else:
        assert not (run / 'last_checkpoint.pth').exists(), '실험 출처 기록이 없는 체크포인트는 자동 재개하지 않습니다.'
    planned_commands[size], planned_setups[size] = command, setup
# Validate both planned runs before writing provenance or starting either run.
for size in (256, 384):
    setup_path = RUNS / f'setup_{size}.json'
    if not setup_path.exists():
        setup_path.write_text(json.dumps(planned_setups[size], ensure_ascii=False, indent=2))
    command = planned_commands[size]
    print(f'시작: {size}px / 5 epoch / lr={LR} / resume={"--resume" in command}', flush=True)
    with (RUNS / f'training_{size}.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    assert digest(SOURCE) == EXPECTED_SOURCE_SHA256
print('256px / 384px 미세조정 완료. 기존 모델 해시 유지 확인.', flush=True)


## 전체 검증 비교
무노균병 재현율·정밀도, 무검은무늬병과의 혼동, 전체 Macro F1을 함께 확인합니다. 단일 분할·seed에서 나온 결과이므로 일반화 개선을 확정하지 않습니다.

In [ ]:
subprocess.run([sys.executable, str(WORK / 'code/compare_resolution_finetune.py'),
    '--control-dir', str(RUNS / 'finetune_256'), '--candidate-dir', str(RUNS / 'finetune_384'),
    '--reference-dir', str(REFERENCE), '--output-dir', str(RUNS / 'comparison')], check=True)
from IPython.display import Markdown, display
display(Markdown((RUNS / 'comparison/comparison.md').read_text(encoding='utf-8')))
print('기존 최고 모델:', SOURCE)
print('256px 대조군:', RUNS / 'finetune_256/best_model.pth')
print('384px 실험군:', RUNS / 'finetune_384/best_model.pth')
